## Data Cleaning Log

### Column selection (before cleaning)

| No. | Check | Reason | Action |
|---|---|---|---|
| A | Host-identifying columns | The project does not need to identify hosts, and these columns (names, IDs, profile URLs, descriptions, locations, pictures) are personal data | Excluded 10 host-identifying columns. |
| B | Other host-related columns | The analysis focuses on listing prices and property characteristics, not on hosts. `host_is_superhost` is an evaluative label about an individual host, so it was removed to stay consistent with the commitment not to profile or evaluate hosts | Excluded 18 other host-related columns, including `host_is_superhost`. |
| C | Remaining unused columns | Listing URLs and descriptions, availability, extra review metrics, price-quote fields and similar are not needed for the initial price comparison | Excluded the remaining unused columns, leaving the 14 columns used in the analysis. 76 of the 90 original columns were excluded in total. |

### Cleaning steps

| No. | Check | Reason | Action |
|---|---|---|---|
| 1 | Duplicate IDs | Duplicate listings would be counted twice and distort price comparisons | No duplicate IDs found. No action needed. |
| 2 | Price format | Prices were stored as text, so they could not be used in the analysis | Removed `$` signs and commas and converted the `price` column to `float64`. Checked 56,873 rows. |
| 3 | Missing prices | A listing without a valid price cannot be compared | Removed rows where `price` was missing after numeric conversion. All 5,009 were already missing in the raw file, so the conversion created no new missing values. 51,864 rows remained. |
| 4 | Bathroom format | `bathrooms_text` holds text such as "1 shared bath", which cannot be used in calculations | Created a numeric `bathrooms` column from `bathrooms_text`, counted after step 3 (51,864 rows). The first number in the text is used, so "3 baths" gives 3, "1 shared bath" gives 1, and "0 baths" gives 0. The 158 values without a number ("Half-bath", "Private half-bath", "Shared half-bath") were set to 0.5. The remaining 60 values could not be converted because `bathrooms_text` itself was missing, so they were kept as NaN. |
| 5 | Amenities | The amenities column holds a list of names as one text string, so it cannot be analyzed directly | Parsed each string as a list with `json.loads` and stored the number of items in `amenities_count`. All 51,864 values parsed without failure. 44 listings have an empty list and received a count of 0. Counts range from 0 to 100. |
| 6 | Missing bedrooms | Deleting these listings would lose about 15% of the rows, and the correct bedroom count cannot be inferred from the other columns | Counted after step 3: 7,859 missing values (8,513 before the missing-price rows were removed). The rows were kept and `bedrooms` was left as NaN. No values were filled in. |
| 7 | Invalid ratings | Ratings are on a 1 to 5 scale, so a value outside that range is invalid | Checked `review_scores_rating` for values below 1 and above 5. Two listings (ids 1084850 and 1322855) had a rating of 0.0. The ratings were set to NaN and both listings were kept. |
| 8 | Missing ratings | A listing with no reviews has no rating, which is different from a rating of 0 | 14,501 ratings are NaN after this step: 14,499 listings with zero reviews (checked: none of them had a rating) plus the 2 values set to NaN in step 7. All of these listings were kept, and only their `review_scores_rating` is empty. |
| 9 | Missing minimum nights | One missing value is too few to justify removing the row, and no justified default exists | Found 1 missing `minimum_nights` value after step 3. The row was kept with NaN. It will need to be excluded or handled explicitly if `minimum_nights` is used later, not replaced with 0. |
| 10 | Long minimum stays | Stays longer than 365 nights are not typical short-term rentals and could distort price comparisons | 4 listings had `minimum_nights` above 365, but all had missing prices and were already removed in step 3. Added `minimum_nights_flag` (True above 365). No remaining rows are flagged. |
| 11 | Geographic coordinates | Listings outside Sicily and its islands would be outside the study area | Range check only: latitude between 35.4 and 38.9 and longitude between 11.8 and 15.7. These limits come from the approximate geography of Sicily and its outlying islands (for example Lampedusa and Pantelleria), not from an official boundary file. In the raw data (56,873 rows), latitude ranged from 35.49 to 38.81 and longitude from 11.93 to 15.65. A bounding box cannot confirm that a point is inside Sicily, since it also covers Malta and the southern tip of Calabria. No rows were outside the range, so none were removed. |
| 12 | High price per guest | Very high prices per guest could distort averages, but the dataset cannot show whether a price was entered wrongly, since large or luxury properties can be genuinely expensive. They are flagged for the later analysis | Calculated `price / accommodates` (daily listing price divided by the number of guests). Because it is right-skewed, the fence was computed on the log10 of price per guest as Q3 + 3 x IQR, which is about 507.97 USD per guest per night, using the 51,864 rows left after step 11. Flagged 40 listings above it in `high_price_per_guest_flag` and kept them. The multiplier 3 is the conventional far-out fence; 1.5 would flag 663 listings. 51,864 rows remained. |
| 13 | Low price per guest | Very low prices per guest are suspicious, but the dataset cannot confirm an error. Some flagged listings are rooms with a capacity of 16 and over 100 reviews, which suggests the price may refer to one bed or room, so they are flagged instead of deleted | Applied the same rule as step 12 at the low end: Q1 - 3 x IQR on the log10 of price per guest, which is about 2.58 USD per guest per night. Flagged 14 listings below it in `low_price_per_guest_flag` and kept them. With a multiplier of 1.5, 557 listings would be flagged. 51,864 rows remained. |
| 14 | Price outliers | Some very high prices are real (such as large properties), so extreme prices are flagged for further investigation instead of being deleted | Calculated price per guest as `price / accommodates`. Because it is right-skewed, Q1, Q3 and IQR were computed on the log10 of price per guest separately for each `room_type`, using the 51,864 rows left after step 11. Flagged 669 listings (about 1.3%) above Q3 + 1.5 x IQR in `price_outlier` and kept them. The fences are about 163.28 USD (Entire home/apt), 167.30 USD (Hotel room), 161.32 USD (Private room) and 179.27 USD (Shared room) per guest per night. The 1.5 multiplier is the conventional IQR rule. All 40 listings flagged in step 12 are also flagged here, since step 12 flags extremes across the whole dataset while this step flags listings that are high compared with their own room type. A flag marks an unusually high price per guest within a room type; it does not prove that the listing is overpriced or that the price is wrong. Comparison across areas is done in the later analysis. |

**Final dataset:** 56,873 rows in, 51,864 rows out (5,009 removed). 14 columns in, 20 out (added bathrooms, amenities_count, minimum_nights_flag, high_price_per_guest_flag, low_price_per_guest_flag, price_outlier).

All flagged listings are kept. The analysis will be run with and without them to check that the results do not depend on them.

In [1]:
derived = [
    "bathrooms",
    "amenities_count",
    "minimum_nights_flag",
    "high_price_per_guest_flag",
    "low_price_per_guest_flag",
    "price_outlier",
]

print("Derived columns:", derived)
print("Missing values in derived columns:")
print(df_clean[derived].isna().sum())
print()
print("Flag counts (True):")
for c in ["minimum_nights_flag", "high_price_per_guest_flag", "low_price_per_guest_flag", "price_outlier"]:
    print(c, int(df_clean[c].sum()))
print()
print(df_clean[["bathrooms", "amenities_count"]].describe())
print()
print(df_clean[["id", "bathrooms_text", "bathrooms", "amenities_count", "price", "accommodates", "price_outlier"]].head(10).to_string(index=False))


Derived columns: ['bathrooms', 'amenities_count', 'minimum_nights_flag', 'high_price_per_guest_flag', 'low_price_per_guest_flag', 'price_outlier']
Missing values in derived columns:
bathrooms                    60
amenities_count               0
minimum_nights_flag           0
high_price_per_guest_flag     0
low_price_per_guest_flag      0
price_outlier                 0
dtype: int64

Flag counts (True):
minimum_nights_flag 0
high_price_per_guest_flag 40
low_price_per_guest_flag 14
price_outlier 669

          bathrooms  amenities_count
count  51804.000000     51864.000000
mean       1.404554        32.000540
std        0.933852        16.525592
min        0.000000         0.000000
25%        1.000000        19.000000
50%        1.000000        32.000000
75%        1.500000        44.000000
max       19.000000       100.000000

    id bathrooms_text  bathrooms  amenities_count  price  accommodates  price_outlier
  7011        3 baths        3.0               48 262.00             6    